<a href="https://colab.research.google.com/github/kimcheol25/food-scanner/blob/main/Colab_%EC%8B%9C%EC%9E%91%ED%95%98%EA%B8%B0.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
# 1. 라이브러리 설치 (코랩에서 최초 1회 실행)
!pip install gradio tensorflow pillow numpy

import gradio as gr
import tensorflow as tf
import numpy as np
from PIL import Image

# 2. 사전 학습된 고성능 AI 모델 로드 (MobileNetV2)
# 데이터셋을 따로 학습시키지 않아도 수천 가지 사물과 식재료를 즉시 인식합니다.
model = tf.keras.applications.MobileNetV2(weights='imagenet')

# 3. 자체 내장 식재료 대용량 데이터베이스 (보관법 및 레시피)
food_database = {
    "육류 (고기류)": {
        "storage": "🥩 [육류 신선 보관 가이드]\n\n• 냉장 보관: 1~2일 이내에 조리할 경우 0~4℃에서 보관하세요.\n• 냉동 보관: 장기 보관 시 1회 분량씩 소분한 후 밀폐 용기나 지퍼백에 넣어 -18℃ 이하로 보관합니다.\n• 신선 꿀팁: 고기 표면에 식용유나 올리브유를 살짝 바르면 수분 증발과 산화를 막아 훨씬 오래 보관할 수 있습니다.",
        "recipes": "🍳 [추천 엄선 레시피]\n\n1. 매콤 제육볶음\n   - 재료: 돼지고기 슬라이스, 양파, 대파, 고추장 양념장\n   - 조리법: 양념에 버무린 고기를 달궈진 팬에 강불로 빠르게 볶아내어 육즙을 잡습니다.\n\n2. 찹스테이크\n   - 소고기와 파프리카를 굴소스와 돈가스 소스 조합으로 볶아내는 초간단 요리.\n\n3. 닭가슴살 아보카도 샐러드"
    },
    "채소류 (야채)": {
        "storage": "🥬 [채소류 신선 보관 가이드]\n\n• 냉장 보관: 잎채소는 씻지 않은 채로 키친타월에 싸서 지퍼백에 보관하면 수분 관리가 잘 됩니다.\n• 실온 보관: 감자, 고구마, 양파는 직사광선이 없는 서늘하고 통풍이 잘되는 곳에 보관하세요. (감자와 양파를 같이 두면 쉽게 무르므로 분리 보관)\n• 신선 꿀팁: 잎채소는 뿌리가 아래로 가도록 세워서 보관하면 신선도가 2배 오래 유지됩니다.",
        "recipes": "🍳 [추천 엄선 레시피]\n\n1. 클래식 라따뚜이\n   - 재료: 가지, 애호박, 토마토, 토마토소스\n   - 조리법: 야채를 얇게 슬라이스하여 원형으로 예쁘게 겹쳐 올린 후 토마토소스를 깔고 오븐이나 팬에 조리합니다.\n\n2. 소고기 채소 짜글이\n3. 모둠 버섯 굴소스 볶음"
    },
    "유제품": {
        "storage": "🧀 [유제품 신선 보관 가이드]\n\n• 냉장 보관: 반드시 0~4℃ 냉장실 안쪽에 보관하세요. 문 쪽은 온도 변화가 심해 피하는 것이 좋습니다.\n• 치즈 보관: 개봉한 치즈는 단면이 마르지 않도록 랩으로 밀착 포장한 후 밀폐용기에 넣어야 밀봉이 유지됩니다.\n• 주의 사항: 우유나 요거트는 주변의 음식 냄새를 쉽게 흡수하므로 반드시 입구를 꽉 닫아 보관하세요.",
        "recipes": "🍳 [추천 엄선 레시피]\n\n1. 치즈 라면 스낵\n   - 재료: 라면, 체다치즈, 설탕 약간\n   - 조리법: 부순 라면을 팬에 굽거나 전자레인지에 돌린 후 체다치즈를 올려 녹여 먹는 초간단 간식.\n\n2. 부드러운 투움바 파스타\n3. 수제 리코타 치즈 샐러드"
    },
    "해산물류": {
        "storage": "🐟 [해산물 신선 보관 가이드]\n\n• 당일 조리: 흐르는 물에 깨끗이 씻어 물기를 완전히 제거한 후 냉장 보관하고 즉시 조리하세요.\n• 장기 보관: 생선은 내장과 비늘을 완벽히 제거하고 소금을 살짝 뿌려 랩으로 감싼 뒤 냉동실에 보관합니다.\n• 주의 사항: 해동한 해산물을 다시 재냉동하면 세균 번식 위험이 매우 높으므로 절대 금지합니다.",
        "recipes": "🍳 [추천 엄선 레시피]\n\n1. 감바스 알 아히요\n   - 재료: 새우, 마늘, 올리브오일, 페페론치노\n   - 조리법: 올리브유에 마늘 향을 충분히 내어준 뒤 새우와 소금을 넣어 익혀줍니다.\n\n2. 칼칼한 바지락 술찜\n3. 매콤달콤 오징어볶음"
    },
    "과일류": {
        "storage": "🍎 [과일류 신선 보관 가이드]\n\n• 후숙 과일: 바나나, 망고, 아보카도는 실온에서 보관하며 알맞게 익은 후 냉장고로 옮깁니다.\n• 일반 과일: 사과는 에틸렌 가스를 방출하여 다른 과일을 쉽게 상하게 하므로, 반드시 비닐팩에 따로 격리하여 보관해야 합니다.\n• 포도는 씻지 않은 상태로 한 송이씩 종이에 싸서 냉장 보관하는 것이 가장 좋습니다.",
        "recipes": "🍳 [추천 엄선 레시피]\n\n1. 프렌치 믹스베리 토스트\n2. 상큼한 요플레 과일 샐러드\n3. 홈메이드 생과일 에이드"
    },
    "기타 식재료": {
        "storage": "📦 [일반 식재료 보관 가이드]\n\n• 직사광선을 피해 서늘하고 건조한 실온에 보관하거나, 개봉 후에는 밀폐하여 냉장 보관하는 것을 권장합니다.",
        "recipes": "🍳 [추천 엄선 레시피]\n\n1. 냉장고 파먹기 만능 볶음밥\n2. 백종원풍 간장 비빔국수"
    }
}

# 4. 핵심 기능: AI 예측 및 페이지 전환 로직
def process_and_navigate(image, category):
    # 사진이 없는 경우 예외 처리
    if image is None:
        return (
            gr.update(), gr.update(),
            "⚠️ 사진을 촬영하거나 업로드해주세요.",
            "카테고리를 먼저 선택해주세요.",
            "결과가 비어있습니다."
        )

    # 1) 이미지 AI 분석 전처리 (224x224 사이즈 조정)
    img = image.resize((224, 224))
    img_array = tf.keras.preprocessing.image.img_to_array(img)
    img_array = tf.keras.applications.mobilenet_v2.preprocess_input(img_array)
    img_array = np.expand_dims(img_array, axis=0)

    # 2) AI 모델 이미지 인식 예측 수행
    predictions = model.predict(img_array)
    decoded_preds = tf.keras.applications.mobilenet_v2.decode_predictions(predictions, top=1)[0]
    predicted_item = decoded_preds[0][1].replace('_', ' ').title()
    confidence = decoded_preds[0][2] * 100

    ai_result_text = f"🤖 **AI 분석 완료**: 촬영된 식재료는 **[{predicted_item}]** 일 확률이 **{confidence:.1f}%** 입니다."

    # 3) 사용자가 선택한 카테고리에 맞는 보관법/레시피 가져오기
    info = food_database.get(category, food_database["기타 식재료"])

    # 4) 화면 제어: 1페이지(입력) 비활성화, 2페이지(결과) 활성화 전달
    return (
        gr.update(visible=False),    # page1 안 보이게 전환
        gr.update(visible=True),     # page2 보이게 전환
        ai_result_text,              # AI 결과 텍스트 반영
        info["storage"],             # 보관법 텍스트 반영
        info["recipes"]              # 레시피 텍스트 반영
    )

# 다시 촬영하기 버튼 클릭 시 기존 입력 화면으로 돌아가는 함수
def reset_and_return():
    return (
        gr.update(visible=True),     # page1 다시 보이게
        gr.update(visible=False),    # page2 숨기기
        None,                        # 기존 이미지 초기화
        "육류 (고기류)"              # 카테고리 초기값 리셋
    )

# 5. 모바일 타겟 디자인 및 레이아웃 구성
# 모바일 브라우저 폭에 최적화되도록 CSS 스타일을 가볍게 추가했습니다.
mobile_css = """
.main_container { max-width: 480px; margin: 0 auto; padding: 10px; background-color: #fcfcfc; }
.btn_primary { background-color: #ff6b6b !important; color: white !important; font-weight: bold; }
.btn_secondary { background-color: #e0e0e0 !important; color: #333 !important; }
"""

with gr.Blocks(theme=gr.themes.Soft(), css=mobile_css) as app:
    with gr.Column(elem_classes="main_container"):

        # ==================== [PAGE 1: 식재료 촬영 및 입력 화면] ====================
        with gr.Column(visible=True) as page1:
            gr.Markdown("<h2 style='text-align: center; color: #ff6b6b; margin-bottom: 0;'>📸 스마트 식재료 스캐너</h2>")
            gr.Markdown("<p style='text-align: center; font-size: 13px; color: #666; margin-top: 5px;'>재료 사진을 찍으면 인공지능이 보관법과 레시피를 알려줍니다.</p>")

            # 모바일 환경에서 접속 시 자동으로 카메라 촬영 연동이 가능한 이미지 컴포넌트
            image_input = gr.Image(type="pil", label="식재료 촬영 (또는 갤러리 업로드)")

            category_input = gr.Dropdown(
                choices=["육류 (고기류)", "채소류 (야채)", "유제품", "해산물류", "과일류", "기타 식재료"],
                label="식재료 분류 선택 (정확도 향상)",
                value="육류 (고기류)"
            )

            submit_btn = gr.Button("🔍 분석 및 레시피 확인하기", elem_classes="btn_primary")

        # ==================== [PAGE 2: AI 분석 및 추천 결과 화면] ====================
        with gr.Column(visible=False) as page2:
            gr.Markdown("<h2 style='text-align: center; color: #4b7bec; margin-bottom: 15px;'>📋 AI 분석 리포트</h2>")

            ai_output = gr.Markdown()

            gr.Markdown("---")
            storage_output = gr.Textbox(label="🧊 신선 보관 가이드", lines=7, interactive=False)
            recipe_output = gr.Textbox(label="👩‍🍳 맞춤 활용 레시피 추천", lines=9, interactive=False)

            gr.Markdown("---")
            back_btn = gr.Button("🔄 다른 재료 다시 촬영하기", elem_classes="btn_secondary")

        # ==================== [이벤트 리스너 및 페이지 내비게이션 연결] ====================
        # 분석하기 버튼 클릭 시 -> 데이터 처리 후 page1 숨기고 page2 띄우기
        submit_btn.click(
            fn=process_and_navigate,
            inputs=[image_input, category_input],
            outputs=[page1, page2, ai_output, storage_output, recipe_output]
        )

        # 다시 촬영하기 버튼 클릭 시 -> 입력을 리셋하고 page1로 돌아가기
        back_btn.click(
            fn=reset_and_return,
            inputs=[],
            outputs=[page1, page2, image_input, category_input]
        )

# 6. 모바일 외부 접속 링크 활성화 (share=True)
app.launch(share=True, debug=True)

14536120/14536120 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


/tmp/ipykernel_1715/3692865426.py:95: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=mobile_css) as app:
/tmp/ipykernel_1715/3692865426.py:95: DeprecationWarning: The 'css' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'css' to Blocks.launch() instead.
  with gr.Blocks(theme=gr.themes.Soft(), css=mobile_css) as app:


Colab notebook detected. This cell will run indefinitely so that you can see errors and logs. To turn off, set debug=False in launch().
* Running on public URL: https://5e2ae55937c0f35f6b.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
